# 跨模态交叉注意力对齐

**学习目标**：实现论文的核心模块——用交叉注意力机制，将 DOM 节点特征与图像 patch 特征对齐。

完成本章后你将掌握：
- 自注意力 vs 交叉注意力的区别
- 用 `nn.MultiheadAttention` 实现交叉注意力
- 边界框 IoU 计算（论文的对齐监督信号）
- 对齐损失函数设计

**输入**（前两章的产出）：
- 视觉特征序列：`[B, 196, 768]`
- 代码特征序列：`[B, N, 768]`

**输出**：
- 每个 DOM 节点对应的视觉区域（注意力权重 `[B, N, 196]`）
- 融合后的双模态特征 `[B, N, 768]`

## Part 1：自注意力 vs 交叉注意力

注意力机制的核心公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **自注意力**：Q、K、V 来自**同一个**序列（序列内部各元素互相关注）
- **交叉注意力**：Q 来自**一个**序列，K、V 来自**另一个**序列（跨序列关注）

在论文中：
- Q = 代码特征（DOM 节点）→ 每个节点去问：「我在图像哪里？」
- K, V = 视觉特征（图像 patch）→ 图像各区域回答：「我长这样」
- 输出 = 每个 DOM 节点融合了它对应视觉区域信息后的新表征

**任务**：用随机数据演示交叉注意力的输入输出形状。

```python
import torch
import torch.nn as nn

B = 2      # batch size
N = 8      # DOM 节点数
P = 196    # 图像 patch 数
D = 768    # 特征维度

visual_features = torch.randn(B, P, D)  # ViT 输出
code_features   = torch.randn(B, N, D)  # CodeBERT 输出

# nn.MultiheadAttention 期望输入格式是 [seq_len, batch, dim]
# 需要先用 .transpose(0, 1) 把 batch 维度和 seq_len 维度交换
cross_attn = nn.MultiheadAttention(embed_dim=D, num_heads=8, batch_first=True)

# Q=代码特征，K=V=视觉特征
# 输出：attn_output [B, N, D]，attn_weights [B, N, P]
attn_output, attn_weights = cross_attn(
    query=...,   # 填入代码特征
    key=...,     # 填入视觉特征
    value=...    # 填入视觉特征
)

print(attn_output.shape)   # 应为 [B, N, D] = [2, 8, 768]
print(attn_weights.shape)  # 应为 [B, N, P] = [2, 8, 196]
```

**思考**：`attn_weights[0, 3]` 是什么含义？它的所有值加起来等于多少？

In [ ]:
# 在这里写代码


## Part 2：IoU 计算 —— 论文的对齐监督信号

交叉注意力需要训练才有意义。论文用**渲染区域 IoU** 作为监督信号：
如果第 i 个代码节点的渲染框 与 第 j 个 patch 的位置框 重叠度（IoU）超过阈值，则认为两者「对齐」。

IoU（Intersection over Union）公式：
$$\text{IoU} = \frac{\text{交集面积}}{\text{并集面积}}$$

边界框格式：`[x1, y1, x2, y2]`（左上角和右下角坐标）

**任务**：实现 `compute_iou(box_a, box_b)` 函数，计算两个边界框的 IoU。

```python
def compute_iou(box_a, box_b):
    """
    box_a, box_b: [x1, y1, x2, y2] 格式的边界框（tensor 或 list）
    返回：IoU 值（0~1 之间的浮点数）
    """
    # 1. 计算交集区域的左上角：取两个框左上角坐标的最大值
    # 2. 计算交集区域的右下角：取两个框右下角坐标的最小值
    # 3. 计算交集面积（注意：如果没有交集，面积为 0，用 clamp(min=0) 处理）
    # 4. 计算两个框各自的面积
    # 5. IoU = 交集面积 / (面积A + 面积B - 交集面积)
    pass

# 验证：
box_a = [0, 0, 100, 100]   # 100x100 的框
box_b = [50, 50, 150, 150] # 与 box_a 有 50x50 的重叠
# 交集 = 50*50 = 2500，并集 = 100*100 + 100*100 - 2500 = 17500
# 期望 IoU ≈ 0.1429
print(compute_iou(box_a, box_b))

box_c = [200, 200, 300, 300]  # 完全不重叠
# 期望 IoU = 0.0
print(compute_iou(box_a, box_c))
```

In [ ]:
# 在这里写代码


## Part 3：构建对齐标签矩阵

有了 IoU，就可以为每对（DOM节点, patch）生成对齐标签：IoU > 阈值 → 1（对齐），否则 → 0（不对齐）。

在实际数据中：
- DOM 节点的边界框来自预处理阶段（Playwright 计算的渲染坐标）
- Patch 的边界框由其在 224x224 图像中的位置决定：第 i 个 patch 的坐标是固定的

ViT patch 坐标规律：图像被均匀切成 14x14 个 16x16 的小块。
第 (row, col) 个 patch 的边界框为 `[col*16, row*16, (col+1)*16, (row+1)*16]`

**任务**：生成所有 196 个 patch 的边界框，再根据模拟的 DOM 节点渲染坐标，计算对齐标签矩阵。

```python
import torch

# 生成 196 个 patch 的边界框 [196, 4]
patch_boxes = []
for row in range(14):
    for col in range(14):
        # 每个 patch 是 16x16 像素
        x1, y1 = col * 16, row * 16
        x2, y2 = x1 + 16, y1 + 16
        patch_boxes.append([x1, y1, x2, y2])
patch_boxes = torch.tensor(patch_boxes, dtype=torch.float32)  # [196, 4]

# 模拟 5 个 DOM 节点的渲染坐标（在 224x224 图像上的位置）
node_boxes = torch.tensor([
    [0,   0,   224, 30 ],  # nav（顶部导航栏）
    [0,   0,   60,  30 ],  # a（nav 内的链接）
    [0,   30,  224, 224],  # main
    [10,  40,  150, 130],  # img
    [10,  140, 110, 170],  # button
], dtype=torch.float32)  # [5, 4]

# 计算对齐标签矩阵 [N, P]：对每对(节点i, patchj)计算IoU，超过阈值则为1
# 提示：用两层循环，调用 Part2 的 compute_iou 函数
IOU_THRESHOLD = 0.3
N = node_boxes.shape[0]
P = patch_boxes.shape[0]
alignment_labels = torch.zeros(N, P)
# ... 填写循环逻辑 ...

print(f'对齐标签矩阵形状: {alignment_labels.shape}')  # [5, 196]
print(f'nav 节点对齐的 patch 数量: {alignment_labels[0].sum().int()}')
print(f'button 节点对齐的 patch 数量: {alignment_labels[4].sum().int()}')
```

**思考**：nav（全宽顶部导航）和 button（小按钮）各自对齐的 patch 数量相差多少？这符合直觉吗？

In [ ]:
# 在这里写代码


## Part 4：可视化对齐结果

把对齐标签矩阵可视化成热力图，直观确认「节点和 patch 的对应关系是否合理」。
这类图可以直接用在论文的实验部分。

**任务**：
1. 把某个节点的注意力权重（或对齐标签）reshape 成 14x14，用热力图展示
2. 同时展示原始图像，对比注意力区域和节点实际位置

```python
import matplotlib.pyplot as plt
import matplotlib.patches as patches

node_names = ['nav', 'a', 'main', 'img', 'button']
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, name in enumerate(node_names):
    # 上排：对齐标签 reshape 成 14x14 热力图
    label_map = alignment_labels[i].reshape(14, 14).numpy()
    axes[0, i].imshow(label_map, cmap='Blues', vmin=0, vmax=1)
    axes[0, i].set_title(f'<{name}> 对齐区域')
    axes[0, i].axis('off')

    # 下排：在 224x224 坐标系中画出节点边界框
    axes[1, i].set_xlim(0, 224)
    axes[1, i].set_ylim(224, 0)  # y 轴翻转，符合图像坐标系
    axes[1, i].set_aspect('equal')
    box = node_boxes[i].numpy()
    rect = patches.Rectangle(
        (box[0], box[1]), box[2]-box[0], box[3]-box[1],
        linewidth=2, edgecolor='red', facecolor='none'
    )
    axes[1, i].add_patch(rect)
    axes[1, i].set_title(f'<{name}> 渲染框')

plt.tight_layout()
plt.show()
```

In [ ]:
# 在这里写代码


## Part 5：对齐损失函数

有了对齐标签，就可以设计损失函数来训练交叉注意力模块。

论文的思路：注意力权重 `attn_weights[i]`（形状 `[196]`）应该在对齐的 patch 上高，在不对齐的 patch 上低。
这本质上是一个**二分类问题**，用 Binary Cross Entropy（BCE）损失：

$$\mathcal{L}_{align} = -\frac{1}{N \cdot P}\sum_{i=1}^{N}\sum_{j=1}^{P} \left[ y_{ij}\log(\hat{y}_{ij}) + (1-y_{ij})\log(1-\hat{y}_{ij}) \right]$$

其中 $y_{ij}$ 是 Part 3 计算的对齐标签，$\hat{y}_{ij}$ 是交叉注意力的权重值。

**任务**：用模拟数据计算一次对齐损失。

```python
import torch.nn as nn

# 模拟交叉注意力输出的注意力权重 [N, P]
# 真实训练中来自 cross_attn() 的第二个返回值
mock_attn_weights = torch.rand(5, 196)  # 随机初始化，训练前没有意义

# BCE 损失：预测权重 vs 对齐标签
loss_fn = nn.BCELoss()
align_loss = loss_fn(mock_attn_weights, alignment_labels)
print(f'对齐损失: {align_loss.item():.4f}')

# 思考：如果注意力权重恰好等于对齐标签（完美对齐），损失是多少？
perfect_weights = alignment_labels.clone()
# 注意：BCE 要求输入在 (0,1) 之间，不能有精确的 0 或 1
perfect_weights = perfect_weights.clamp(1e-6, 1 - 1e-6)
perfect_loss = loss_fn(perfect_weights, alignment_labels)
print(f'完美对齐时的损失: {perfect_loss.item():.4f}')
```

**思考**：随机初始化时损失约为多少？完美对齐时约为多少？两者差距体现了什么？

In [ ]:
# 在这里写代码


## Part 6：整合 —— 完整的对齐模块

把前五个 Part 整合成一个 `CrossModalAlignment` 模块，这是论文代码的第一个正式 `nn.Module`。

**任务**：完成下面的类，使其能完整跑通前向传播和损失计算。

```python
import torch
import torch.nn as nn

class CrossModalAlignment(nn.Module):
    def __init__(self, dim=768, num_heads=8):
        super().__init__()
        # 初始化交叉注意力层
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=num_heads, batch_first=True
        )
        self.loss_fn = nn.BCELoss()

    def forward(self, visual_features, code_features, alignment_labels=None):
        """
        visual_features:   [B, 196, 768] ViT 输出
        code_features:     [B, N,   768] CodeBERT 输出
        alignment_labels:  [B, N,   196] IoU 对齐标签（训练时传入，推理时为 None）

        返回：
          fused_features: [B, N, 768]  融合后的双模态特征
          attn_weights:   [B, N, 196]  注意力权重（可用于可视化）
          loss:           标量（训练时返回，推理时返回 None）
        """
        # 1. 交叉注意力：Q=code, K=V=visual
        # 2. 如果有 alignment_labels，计算 BCE 损失
        # 3. 返回 fused_features, attn_weights, loss
        pass

# 测试
model = CrossModalAlignment()
visual = torch.randn(2, 196, 768)
code   = torch.randn(2, 8, 768)
labels = torch.randint(0, 2, (2, 8, 196)).float()

fused, weights, loss = model(visual, code, labels)
print(f'融合特征: {fused.shape}')    # [2, 8, 768]
print(f'注意力权重: {weights.shape}') # [2, 8, 196]
print(f'对齐损失: {loss.item():.4f}')
```

**完成标志**：三个 print 输出正确，这就是论文对齐模块的完整实现骨架。

In [ ]:
# 在这里写代码
